# Py4J Bridge Demo

This notebook exercises the full Lumos cross-boundary chain:

**notebook cell  -->  Python wrapper  -->  Scala class (via Py4J / JVM)**

After scanning, you should see these edges in the knowledge graph:
- `cross_boundary_import` — notebook imports `FeatureEngineerWrapper`
- `cross_boundary_call` — notebook calls `wrapper.compute_metrics(...)`
- `py4j_bridge` — wrapper instantiates `com.example.MetricsCalculator`
- `py4j_method_call` — wrapper calls `MetricsCalculator.giniCoefficient(...)`
- `py4j_method_call` (notebook-origin) — Cell 4 calls Scala directly without the wrapper

## 1. Setup

Create a SparkSession and import the Python wrapper that fronts the Scala implementation.

In [1]:
from pyspark.sql import SparkSession
from spark_demo.python_wrapper import FeatureEngineerWrapper

spark = SparkSession.builder.appName("py4j-demo").getOrCreate()

## 2. Use the wrapper (recommended pattern)

The wrapper hides the JVM gateway behind a clean Python API. Lumos traces through the
wrapper to the underlying Scala class.

In [2]:
predictions = spark.createDataFrame(
    [(0, 0.1), (1, 0.9), (1, 0.85), (0, 0.2)],
    ["label", "prediction"],
)

wrapper = FeatureEngineerWrapper(spark)
gini = wrapper.compute_metrics(predictions)
print(f"Gini coefficient: {gini}")

In [3]:
evaluation = wrapper.evaluate(predictions)
print(f"Evaluation: {evaluation}")

## 3. Direct Py4J call from the notebook (quick & dirty pattern)

Sometimes you want to invoke a Scala method ad-hoc without writing a wrapper. This
produces a `py4j_method_call` edge whose source is the notebook cell itself, not
a Python file.

In [4]:
# Direct call into Scala — no Python wrapper involved
gini_direct = spark._jvm.com.example.MetricsCalculator.giniCoefficient(
    predictions._jdf, "label", "prediction"
)
print(f"Gini (direct from notebook): {gini_direct}")

## 4. Framework call (should be skipped by Lumos)

Calls to `org.apache.spark.*`, `java.*`, `org.apache.hadoop.*` are framework internals.
Lumos detects but does NOT emit cross-boundary edges for these (avoids noise).

In [5]:
# This is a framework call — Lumos recognizes it as not user code
rdd_class = spark._jvm.org.apache.spark.rdd.RDD
print(rdd_class)